# Week 2: ML Task Framing

## Section 1: My Lane as an ML Task
**Task Type:** Ranking / Scoring

**Framing:**
The goal of this task is to score every page on a website based on its likelihood and severity of traffic/value decline, producing a prioritized list (queue) for human content reviewers. Under the hood, this is framed as a **scoring model** that outputs a probability/urgency score between 0.0 and 1.0 for each page. Sorting the pages by this score produces a **ranked list**, ensuring reviewers address the highest-risk pages first.

## Section 2: Target or Proxy
**Target vs. Proxy:**
- **Real Target:** Content decay that actively degrades user experience and loses business value/relevance.
- **Proxy Label:**  (derived from historical page metrics).

**Why it is a proxy:**
We cannot directly observe "content decay" in raw logs—only traffic changes.  serves as a proxy because traffic drops can happen due to external noise like seasonality (e.g., winter products dropping in spring), short-term algorithm volatility, or tracking glitches—even when page quality remains completely intact.

## Section 3: Success Metric
**Primary Metric:** Precision@K (e.g., Precision@10 or Precision@20)
**Secondary Metric:** PR-AUC (Precision-Recall Area Under the Curve)

**Justification:**
Content reviewer capacity is limited. A **False Positive (FP)** wastes valuable human time auditing healthy pages. A **False Negative (FN)** leaves a declining page undetected, but it can be caught in subsequent runs as traffic drops further.

Because reviewers can only inspect a fixed budget of $ pages per batch, **Precision@K** measures what percentage of the top $ flagged pages actually need intervention. We track **PR-AUC** across the whole dataset instead of ROC-AUC or Accuracy because declining pages are relatively rare (class imbalance).

## Section 4: The Unit of Analysis, as a Real Dataframe
**Unit of Analysis:**
**One row = One unique URL / Page** over a specific evaluation window (e.g., weekly snapshot).

In [ ]:
import pandas as pd
import numpy as np

# Demonstration showing unit of analysis: One row per page
sample_data = {
    "url_id": ["page_101", "page_102", "page_103", "page_104"],
    "url_path": ["/blog/seo-guide", "/pricing", "/docs/api", "/blog/winter-trends"],
    "clicks_last_30d": [1250, 4300, 890, 150],
    "clicks_prev_30d": [1800, 4250, 1200, 1100],
    "traffic_change_pct": [-0.305, 0.011, -0.258, -0.863],
    "trend_direction": ["down", "up", "down", "down"],
    "target_decline_proxy": [1, 0, 1, 1]
}

df_unit_of_analysis = pd.DataFrame(sample_data)
print(f"Dataset Shape: {df_unit_of_analysis.shape}")
display(df_unit_of_analysis.head())

## Section 5: Why ML Beats a Fixed Rule Here
**Why ML beats a fixed heuristic (e.g., "Flag if traffic drops > 20%"):**

1. **Context & Base-Volume Awareness:** A 20% drop on a high-traffic page (losing 10,000 clicks) is a critical business issue. A 20% drop on a low-traffic page (losing 2 clicks) is statistical noise. Hardcoded rules trigger false alarms on low-volume pages while missing high-impact subtle drops.
2. **Multi-Factor Interaction:** Traffic decline depends on a combination of signals—search impressions, click-through rates (CTR), seasonal patterns, and page category. Hardcoded IF/ELSE rules quickly become an unmaintainable web of edge cases. ML automatically learns non-linear combinations of these features.
3. **Adaptability:** A static threshold breaks when baseline site traffic shifts across seasons. ML models re-rank dynamically based on relative feature importances rather than rigid cutoffs.

## Section 6: Self-Check
- [x] **Task Type:** Defined as Ranking / Scoring.
- [x] **Target/Proxy:** Identified real target vs. proxy label ().
- [x] **Success Metric:** Selected **Precision@K** and **PR-AUC**.
- [x] **Unit of Analysis:** Verified 1 row = 1 unique page with executable dataframe output.
- [x] **ML vs. Rule:** Explained why ML beats static percentage thresholds.